# Challenge 1 — Results Analysis

This notebook:
1. Loads the evaluation metrics produced by `evaluate.py` / `compare_models.py`
2. Visualises BLEU / chrF2 / TER across all three test sets
3. Performs error analysis on the custom dataset
4. Shows qualitative examples (low-scoring vs high-scoring sentences)

In [ ]:
import json, pathlib
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

RESULTS_DIR = pathlib.Path('../results')

# Load summary JSONs
baseline_json  = RESULTS_DIR / 'baseline'  / 'metrics_summary.json'
finetuned_json = RESULTS_DIR / 'finetuned' / 'metrics_summary.json'

with open(baseline_json)  as f: baseline  = json.load(f)['results']
with open(finetuned_json) as f: finetuned = json.load(f)['results']

splits  = list(baseline.keys())
metrics = ['BLEU', 'chrF2', 'TER']
print('Splits  :', splits)
print('Baseline:', baseline)
print('FT      :', finetuned)

## 1. Corpus-level metric comparison

In [ ]:
rows = []
for split in splits:
    for metric in metrics:
        rows.append(dict(
            split=split, metric=metric,
            baseline=baseline[split][metric],
            finetuned=finetuned[split][metric],
            delta=round(finetuned[split][metric] - baseline[split][metric], 2)
        ))
comp_df = pd.DataFrame(rows)
comp_df

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
colors = {'Baseline': '#4C72B0', 'Fine-tuned': '#DD8452'}
w = 0.35

for ax, metric in zip(axes, metrics):
    sub = comp_df[comp_df['metric'] == metric]
    x = range(len(sub))
    ax.bar([i - w/2 for i in x], sub['baseline'],  w, label='Baseline',   color=colors['Baseline'])
    ax.bar([i + w/2 for i in x], sub['finetuned'], w, label='Fine-tuned', color=colors['Fine-tuned'])
    ax.set_title(metric, fontsize=13, fontweight='bold')
    ax.set_xticks(list(x))
    ax.set_xticklabels(sub['split'].tolist(), rotation=15, ha='right', fontsize=9)
    ax.legend()
    note = '(lower=better)' if metric == 'TER' else '(higher=better)'
    ax.set_ylabel(f'Score {note}')

fig.suptitle('Baseline vs Fine-tuned: EN→NL Translation Quality', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'metrics_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Per-sentence error analysis — Custom Challenge 1 dataset

In [ ]:
# Load per-sentence detail from fine-tuned model on custom set
detail_path = RESULTS_DIR / 'finetuned' / 'Custom_Challenge1_detail.tsv'
detail_df   = pd.read_csv(detail_path, sep='\t')

print(f'Sentences: {len(detail_df)}')
print(f"Avg BLEU : {detail_df['sent_BLEU'].mean():.2f}")
print(f"Avg chrF2: {detail_df['sent_chrF2'].mean():.2f}")
detail_df[['sent_BLEU','sent_chrF2']].describe()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(detail_df['sent_BLEU'],  bins=20, color='#4C72B0', edgecolor='white')
axes[0].set_title('Sentence-level BLEU distribution')
axes[0].set_xlabel('Sentence BLEU')

axes[1].hist(detail_df['sent_chrF2'], bins=20, color='#DD8452', edgecolor='white')
axes[1].set_title('Sentence-level chrF2 distribution')
axes[1].set_xlabel('Sentence chrF2')

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'sentence_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Qualitative examples

In [ ]:
# Worst-performing sentences (low BLEU)
worst = detail_df.nsmallest(5, 'sent_BLEU')[['source','hypothesis','reference','sent_BLEU','sent_chrF2']]
print('=== WORST (low BLEU) ===')
for _, row in worst.iterrows():
    print(f"  SRC : {row['source']}")
    print(f"  HYP : {row['hypothesis']}")
    print(f"  REF : {row['reference']}")
    print(f"  BLEU: {row['sent_BLEU']}  chrF2: {row['sent_chrF2']}")
    print()

In [ ]:
# Best-performing sentences
best = detail_df.nlargest(5, 'sent_BLEU')[['source','hypothesis','reference','sent_BLEU','sent_chrF2']]
print('=== BEST (high BLEU) ===')
for _, row in best.iterrows():
    print(f"  SRC : {row['source']}")
    print(f"  HYP : {row['hypothesis']}")
    print(f"  REF : {row['reference']}")
    print(f"  BLEU: {row['sent_BLEU']}  chrF2: {row['sent_chrF2']}")
    print()

## 4. Score scatter plot: Sentence length vs BLEU

In [ ]:
detail_df['src_len'] = detail_df['source'].str.split().str.len()

plt.figure(figsize=(8, 5))
plt.scatter(detail_df['src_len'], detail_df['sent_BLEU'], alpha=0.5, color='#4C72B0')
plt.xlabel('Source sentence length (words)')
plt.ylabel('Sentence BLEU')
plt.title('Does sentence length correlate with translation quality?')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'length_vs_bleu.png', dpi=150, bbox_inches='tight')
plt.show()

corr = detail_df[['src_len','sent_BLEU']].corr().iloc[0,1]
print(f'Pearson correlation (length vs BLEU): {corr:.3f}')